> **데이터셋 안내** — 이 노트북이 참조하는 HF 데이터셋은 공개 배포하지 않는다.
> AI Hub 원본에서 재생성하는 절차는 [docs/data/data-pipeline.md](../docs/data/data-pipeline.md)「가공 데이터셋은 배포하지 않는다 — 재현 경로」에 있다.

- `10_03`에서 확인한 제거 대상 샘플을 kobert tokenized 데이터셋에서 제거하고 새로 hf repo에 푸쉬한다. 

## Import, Config

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
os.environ["HF_HOME"] = str(ROOT / ".hf_cache")
from datasets import load_dataset, DatasetDict

In [2]:
config = {
    "tokenized_repo": "ingyoun/patent-clean-text-kobert-tokenized",  # 정리 대상(정리 이전 상태)
    "ref_base": "ingyoun/patent-clean-text",                         # 이미 정리된 원본 — 문서 집합·순서 대조 기준
    "removed_json": "label_conflict_docs.json",                      # 10_03 제거 목록 SSOT
    "out_path": ROOT / "output",
}
DRY_RUN = False    # 검토 후 False로 실행해 Hub 반영
OUT = config["out_path"]
SPLITS = ("train", "val", "test")

## kobert-tokenized 데이터셋 확인

In [3]:
tok = {sp: load_dataset(config["tokenized_repo"], split=sp) for sp in SPLITS}
assert "document_id" in tok["train"].column_names, "토큰화본에 document_id 없음"

# kobert 토큰본은 원문 필드가 없고(document_id/input_ids/attention_mask/labels) 재토큰화도 하지 않는다.
# 10_03이 modernbert 토큰본에 한 것과 동일하게 document_id로 행만 필터링한다.
print("columns:", tok["train"].column_names)
print(f"{'split':6}{'n(정리 이전)':>14}")
for sp in SPLITS:
    print(f"{sp:6}{len(tok[sp]):>14,}")

columns: ['document_id', 'input_ids', 'attention_mask', 'labels']
split       n(정리 이전)
train        201,895
val           11,162
test          11,271


## 제거 대상 샘플 확인

In [4]:
# 현 patent-clean-text(원본)는 이미 정리돼 재검출 시 0건이므로, 제거 목록은 10_03 SSOT에서 로드한다.
rec = json.loads((OUT / config["removed_json"]).read_text(encoding="utf-8"))
remove = set(rec["conflict_docs"] + rec["train_dedup_docs"] + rec["eval_leak_docs"] + rec["valtest_dedup_docs"])

print("정책:", rec["policy"])
print(f"  라벨충돌 {len(rec['conflict_docs'])} · train중복 {len(rec['train_dedup_docs'])}"
      f" · eval누수 {len(rec['eval_leak_docs'])} · val+test {len(rec['valtest_dedup_docs'])}")
print(f"  총 제거 {len(remove)}  ·  SSOT split 분포 {rec['removed_split_dist']}")
assert len(remove) == rec["n_removed_total"] == 336

정책: 라벨충돌=그룹전원제거 · train포함정확중복=train1개유지후나머지제거 · val+test정확중복=test1개유지후val제거 · val전용중복=유지
  라벨충돌 89 · train중복 199 · eval누수 45 · val+test 3
  총 제거 336  ·  SSOT split 분포 {'train': 279, 'test': 27, 'val': 30}


In [5]:
# 336 제거 대상이 kobert 토큰본에 실제 존재하고 split 분포가 10_03과 일치하는지 확인
kids = {sp: set(tok[sp]["document_id"]) for sp in SPLITS}
dist = {sp: sum(d in kids[sp] for d in remove) for sp in SPLITS}
print("kobert 내 제거 대상 split 분포:", dist)
assert dist == rec["removed_split_dist"], "split 분포가 10_03 SSOT와 불일치"
assert sum(dist.values()) == len(remove), "제거 대상 일부가 kobert에 없음"
print("확인: 336 제거 대상 전원 kobert에 존재 · split 분포 SSOT 일치")

kobert 내 제거 대상 split 분포: {'train': 279, 'val': 30, 'test': 27}
확인: 336 제거 대상 전원 kobert에 존재 · split 분포 SSOT 일치


## 중복.충돌 샘플 제거

In [6]:
clean_tok = DatasetDict({sp: tok[sp].filter(lambda ex: ex["document_id"] not in remove) for sp in SPLITS})

print(f"{'split':6}{'before':>9}{'after':>9}{'removed':>9}")
for sp in SPLITS:
    print(f"{sp:6}{len(tok[sp]):>9,}{len(clean_tok[sp]):>9,}{len(tok[sp]) - len(clean_tok[sp]):>9}")

Filter:   0%|          | 0/201895 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11162 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11271 [00:00<?, ? examples/s]

split    before    after  removed
train   201,895  201,616      279
val      11,162   11,132       30
test     11,271   11,244       27


In [7]:
# 정리된 base(patent-clean-text, 336 반영 완료)와 document_id 집합·순서 대조.
# kobert 토큰본엔 원문 필드가 없어 잔여 충돌을 직접 재검출할 수 없으므로,
# 잔여 충돌 0이 확인된 base와 대조해 동일 정화 결과임을 보인다.
base = {sp: load_dataset(config["ref_base"], split=sp) for sp in SPLITS}
TARGET = {"train": 201_616, "val": 11_132, "test": 11_244}
for sp in SPLITS:
    assert len(clean_tok[sp]) == TARGET[sp], (sp, len(clean_tok[sp]))
    assert clean_tok[sp]["document_id"] == base[sp]["document_id"], f"{sp}: 정리 kobert ↔ base 문서·순서 불일치"
print("verify: 정리 크기 == 목표(201,616 / 11,132 / 11,244)")
print("verify: 정리 kobert document_id == base (문서 집합·행 순서 동일, 3 split)")

verify: 정리 크기 == 목표(201,616 / 11,132 / 11,244)
verify: 정리 kobert document_id == base (문서 집합·행 순서 동일, 3 split)


## Push

In [ ]:
# hf push — DRY_RUN 게이트. 검토 후 DRY_RUN=False로 실행해 반영한다.
if DRY_RUN:
    print("[DRY_RUN] push 생략")
    print("  token:", config["tokenized_repo"], {sp: len(clean_tok[sp]) for sp in SPLITS})
else:
    clean_tok.push_to_hub(config["tokenized_repo"])
    print("pushed:", config["tokenized_repo"])